# Semantic search baseline locale

Questo notebook usa gli embeddings gia' salvati per fare semantic search locale.

Input:
- `data/processed/corpus_embeddings.jsonl`
- `data/processed/query_embeddings.jsonl`

Output:
- `data/processed/semantic_search_results.jsonl`
- `data/processed/semantic_search_metrics.json`

Non vengono usati vector database, BM25, hybrid search o FastAPI: il ranking e' calcolato in memoria con cosine similarity e NumPy.

## 1. Configurazione

I percorsi e `top_k` sono configurabili qui oppure tramite variabili d'ambiente.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np


current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
DATA_DIR = PROJECT_ROOT / "data" / "processed"


def configured_path(env_var: str, default_path: Path) -> Path:
    value = os.getenv(env_var)
    path = Path(value) if value else default_path
    return path if path.is_absolute() else PROJECT_ROOT / path


CORPUS_INPUT_PATH = configured_path("SEMANTIC_CORPUS_EMBEDDINGS_PATH", DATA_DIR / "corpus_embeddings.jsonl")
QUERY_INPUT_PATH = configured_path("SEMANTIC_QUERY_EMBEDDINGS_PATH", DATA_DIR / "query_embeddings.jsonl")
RESULTS_OUTPUT_PATH = configured_path("SEMANTIC_SEARCH_RESULTS_PATH", DATA_DIR / "semantic_search_results.jsonl")
METRICS_OUTPUT_PATH = configured_path("SEMANTIC_SEARCH_METRICS_PATH", DATA_DIR / "semantic_search_metrics.json")

TOP_K = int(os.getenv("SEMANTIC_SEARCH_TOP_K", "5"))
if TOP_K <= 0:
    raise ValueError("SEMANTIC_SEARCH_TOP_K deve essere maggiore di zero")

# Le metriche didattiche richiedono top-5. Se TOP_K e' piu' piccolo,
# salviamo comunque solo TOP_K candidati ma valutiamo fino a 5.
EVALUATION_KS = [1, 3, 5]
RETRIEVAL_K = max(TOP_K, max(EVALUATION_KS))

CORPUS_INPUT_PATH, QUERY_INPUT_PATH, RESULTS_OUTPUT_PATH, METRICS_OUTPUT_PATH, TOP_K

## 2. Funzioni semplici

Le funzioni sotto leggono JSONL, controllano i record e salvano output in modo atomico.

In [ ]:
REQUIRED_FIELDS = ["record_id", "source_image", "metadata", "embedding"]


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"File non trovato: {path}")

    records = []
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON non valido in {path}, riga {line_number}") from exc
    return records


def write_jsonl_atomic(records: list[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    tmp_path.replace(path)


def write_json_atomic(data: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.write("\n")
    tmp_path.replace(path)


def duplicate_values(values: list[Any]) -> list[Any]:
    counts = Counter(values)
    return sorted([value for value, count in counts.items() if count > 1])


def validate_records(records: list[dict[str, Any]], name: str) -> None:
    if not records:
        raise ValueError(f"{name}: nessun record trovato")

    for index, record in enumerate(records):
        missing = [field for field in REQUIRED_FIELDS if field not in record]
        if missing:
            raise ValueError(f"{name}: record {index} senza campi obbligatori {missing}")
        if not isinstance(record["embedding"], list) or not record["embedding"]:
            raise ValueError(f"{name}: record {index} ha embedding vuoto o non valido")

    duplicated_ids = duplicate_values([record["record_id"] for record in records])
    if duplicated_ids:
        preview = duplicated_ids[:10]
        raise ValueError(f"{name}: record_id duplicati trovati: {preview}")


def build_embedding_matrix(records: list[dict[str, Any]], name: str) -> np.ndarray:
    dimensions = [len(record["embedding"]) for record in records]
    unique_dimensions = sorted(set(dimensions))
    if len(unique_dimensions) != 1:
        raise ValueError(f"{name}: dimensioni embedding non coerenti: {unique_dimensions[:10]}")

    declared_dimensions = [record.get("embedding_dimension") for record in records if record.get("embedding_dimension") is not None]
    wrong_declared = [declared for declared in declared_dimensions if declared != unique_dimensions[0]]
    if wrong_declared:
        raise ValueError(f"{name}: embedding_dimension non corrisponde alla lunghezza del vettore")

    matrix = np.array([record["embedding"] for record in records], dtype=np.float32)
    if not np.isfinite(matrix).all():
        raise ValueError(f"{name}: trovati valori NaN o inf negli embeddings")

    zero_vectors = int((np.linalg.norm(matrix, axis=1) == 0).sum())
    if zero_vectors:
        raise ValueError(f"{name}: trovati {zero_vectors} vettori con norma zero")

    return matrix


def normalize_rows(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / norms


def essential_metadata(metadata: dict[str, Any]) -> dict[str, Any]:
    if not isinstance(metadata, dict):
        return {}

    keys_to_keep = [
        "evaluation_split",
        "noise_level",
        "error_types",
        "changed_fields",
        "buyer",
        "seller",
        "supplier",
        "invoice",
        "payment",
        "tax",
    ]
    compact = {key: metadata[key] for key in keys_to_keep if key in metadata}

    products = metadata.get("products")
    if isinstance(products, list):
        compact["product_count"] = len(products)

    return compact

## 3. Lettura e controlli

Qui controlliamo record processati, duplicati, dimensioni dei vettori e query senza match nel corpus.

In [ ]:
corpus_records = read_jsonl(CORPUS_INPUT_PATH)
query_records = read_jsonl(QUERY_INPUT_PATH)

validate_records(corpus_records, "corpus")
validate_records(query_records, "queries")

corpus_matrix = build_embedding_matrix(corpus_records, "corpus")
query_matrix = build_embedding_matrix(query_records, "queries")

if corpus_matrix.shape[1] != query_matrix.shape[1]:
    raise ValueError(
        f"Dimensioni diverse: corpus={corpus_matrix.shape[1]}, queries={query_matrix.shape[1]}"
    )

corpus_ids = {record["record_id"] for record in corpus_records}
query_ids = {record["record_id"] for record in query_records}
queries_without_match = sorted(query_ids - corpus_ids)

print(f"Corpus records: {len(corpus_records):,}")
print(f"Query records: {len(query_records):,}")
print(f"Embedding dimension: {corpus_matrix.shape[1]}")
print(f"Query senza match nel corpus: {len(queries_without_match):,}")

if queries_without_match[:10]:
    print("Esempi query senza match:", queries_without_match[:10])

## 4. Semantic search con cosine similarity

Normalizziamo i vettori e poi il prodotto matrice-vettore equivale alla cosine similarity.

In [ ]:
corpus_normalized = normalize_rows(corpus_matrix)
query_normalized = normalize_rows(query_matrix)

# Matrice: righe = queries, colonne = record del corpus.
similarities = query_normalized @ corpus_normalized.T
similarities.shape

## 5. Ranking candidati

Per ogni query salviamo i primi `TOP_K` candidati. Per le metriche usiamo il rank del record corretto, definito da `query.record_id == corpus.record_id`.

In [ ]:
corpus_index_by_record_id = {
    record["record_id"]: index
    for index, record in enumerate(corpus_records)
}

results = []

for query_index, query_record in enumerate(query_records):
    query_scores = similarities[query_index]
    sorted_indices = np.argsort(-query_scores)
    candidate_indices = sorted_indices[:TOP_K]

    correct_corpus_index = corpus_index_by_record_id.get(query_record["record_id"])
    if correct_corpus_index is None:
        correct_rank = None
        reciprocal_rank = 0.0
    else:
        correct_score = query_scores[correct_corpus_index]
        # Rank 1-based. Con eventuali pari merito, contiamo i candidati con score strettamente maggiore.
        correct_rank = int(1 + np.sum(query_scores > correct_score))
        reciprocal_rank = 1.0 / correct_rank

    candidates = []
    for corpus_index in candidate_indices:
        corpus_record = corpus_records[int(corpus_index)]
        candidates.append(
            {
                "record_id": corpus_record["record_id"],
                "source_image": corpus_record.get("source_image"),
                "similarity_score": float(query_scores[corpus_index]),
                "metadata": essential_metadata(corpus_record.get("metadata", {})),
            }
        )

    metadata = query_record.get("metadata", {})
    results.append(
        {
            "record_id": query_record["record_id"],
            "source_image": query_record.get("source_image"),
            "metadata": metadata,
            "evaluation_split": metadata.get("evaluation_split") if isinstance(metadata, dict) else None,
            "correct_rank": correct_rank,
            "reciprocal_rank": reciprocal_rank,
            "is_match_top_1": correct_rank is not None and correct_rank <= 1,
            "is_match_top_3": correct_rank is not None and correct_rank <= 3,
            "is_match_top_5": correct_rank is not None and correct_rank <= 5,
            "candidates": candidates,
        }
    )

print(f"Risultati creati: {len(results):,}")
results[0]

## 6. Metriche didattiche

Calcoliamo top-1 accuracy, top-3 accuracy, top-5 accuracy e MRR sia sul totale sia per `evaluation_split`, se presente nei metadata delle queries.

In [ ]:
def compute_metrics(rows: list[dict[str, Any]]) -> dict[str, Any]:
    if not rows:
        return {
            "query_count": 0,
            "top_1_accuracy": None,
            "top_3_accuracy": None,
            "top_5_accuracy": None,
            "mrr": None,
        }

    return {
        "query_count": len(rows),
        "top_1_accuracy": float(np.mean([row["is_match_top_1"] for row in rows])),
        "top_3_accuracy": float(np.mean([row["is_match_top_3"] for row in rows])),
        "top_5_accuracy": float(np.mean([row["is_match_top_5"] for row in rows])),
        "mrr": float(np.mean([row["reciprocal_rank"] for row in rows])),
    }


split_names = sorted(
    {
        row.get("evaluation_split")
        for row in results
        if row.get("evaluation_split") is not None
    }
)

metrics = {
    "config": {
        "top_k_saved_candidates": TOP_K,
        "evaluation_ks": EVALUATION_KS,
        "corpus_input_path": str(CORPUS_INPUT_PATH),
        "query_input_path": str(QUERY_INPUT_PATH),
        "results_output_path": str(RESULTS_OUTPUT_PATH),
        "metrics_output_path": str(METRICS_OUTPUT_PATH),
    },
    "checks": {
        "corpus_records": len(corpus_records),
        "query_records": len(query_records),
        "embedding_dimension": int(corpus_matrix.shape[1]),
        "duplicate_record_ids_checked": True,
        "queries_without_match_in_corpus": len(queries_without_match),
        "queries_without_match_examples": queries_without_match[:10],
    },
    "overall": compute_metrics(results),
    "by_evaluation_split": {
        split: compute_metrics([row for row in results if row.get("evaluation_split") == split])
        for split in split_names
    },
}

metrics

## 7. Salvataggio output

Il salvataggio e' atomico: ogni riesecuzione sostituisce i file finali e non aggiunge righe duplicate.

In [ ]:
write_jsonl_atomic(results, RESULTS_OUTPUT_PATH)
write_json_atomic(metrics, METRICS_OUTPUT_PATH)

print(f"Salvati risultati: {RESULTS_OUTPUT_PATH}")
print(f"Salvate metriche: {METRICS_OUTPUT_PATH}")

## 8. Lettura rapida delle metriche salvate

In [ ]:
with METRICS_OUTPUT_PATH.open("r", encoding="utf-8") as f:
    saved_metrics = json.load(f)

saved_metrics["overall"], saved_metrics["by_evaluation_split"]